# Importing Dependencies

In [1]:
import numpy as np
import pandas as pd
import re
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score
from imblearn.over_sampling import SMOTE  # Importing SMOTE for oversampling


In [2]:
import nltk
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [3]:
print(stopwords.words('english'))

['i', 'me', 'my', 'myself', 'we', 'our', 'ours', 'ourselves', 'you', "you're", "you've", "you'll", "you'd", 'your', 'yours', 'yourself', 'yourselves', 'he', 'him', 'his', 'himself', 'she', "she's", 'her', 'hers', 'herself', 'it', "it's", 'its', 'itself', 'they', 'them', 'their', 'theirs', 'themselves', 'what', 'which', 'who', 'whom', 'this', 'that', "that'll", 'these', 'those', 'am', 'is', 'are', 'was', 'were', 'be', 'been', 'being', 'have', 'has', 'had', 'having', 'do', 'does', 'did', 'doing', 'a', 'an', 'the', 'and', 'but', 'if', 'or', 'because', 'as', 'until', 'while', 'of', 'at', 'by', 'for', 'with', 'about', 'against', 'between', 'into', 'through', 'during', 'before', 'after', 'above', 'below', 'to', 'from', 'up', 'down', 'in', 'out', 'on', 'off', 'over', 'under', 'again', 'further', 'then', 'once', 'here', 'there', 'when', 'where', 'why', 'how', 'all', 'any', 'both', 'each', 'few', 'more', 'most', 'other', 'some', 'such', 'no', 'nor', 'not', 'only', 'own', 'same', 'so', 'than', '

# Dataset

In [4]:
df = pd.read_csv('/content/PhiUSIIL_Phishing_URL_Dataset (1).csv')

In [5]:
df.head()

,FILENAME,URL,URLLength,Domain,DomainLength,IsDomainIP,TLD,URLSimilarityIndex,CharContinuationRate,TLDLegitimateProb,...,Pay,Crypto,HasCopyrightInfo,NoOfImage,NoOfCSS,NoOfJS,NoOfSelfRef,NoOfEmptyRef,NoOfExternalRef,label
0,521848.txt,https://www.southbankmosaics.com,31,www.southbankmosaics.com,24,0,com,100.0,1.000000,0.522907,...,0,0,1,34,20,28,119,0,124,1
1,31372.txt,https://www.uni-mainz.de,23,www.uni-mainz.de,16,0,de,100.0,0.666667,0.032650,...,0,0,1,50,9,8,39,0,217,1
2,597387.txt,https://www.voicefmradio.co.uk,29,www.voicefmradio.co.uk,22,0,uk,100.0,0.866667,0.028555,...,0,0,1,10,2,7,42,2,5,1
3,554095.txt,https://www.sfnmjournal.com,26,www.sfnmjournal.com,19,0,com,100.0,1.000000,0.522907,...,1,1,1,3,27,15,22,1,31,1
4,151578.txt,https://www.rewildingargentina.org,33,www.rewildingargentina.org,26,0,org,100.0,1.000000,0.079963,...,1,0,1,244,15,34,72,1,85,1


In [6]:
df = df[['URL', 'label']]

In [7]:
df.head()

,URL,label
0,https://www.southbankmosaics.com,1
1,https://www.uni-mainz.de,1
2,https://www.voicefmradio.co.uk,1
3,https://www.sfnmjournal.com,1
4,https://www.rewildingargentina.org,1


In [8]:
df.shape

(235795, 2)

In [9]:
df['label'].value_counts()

,count
label,
1,134850
0,100945


In [10]:
df.isnull().sum()

,0
URL,0
label,0


In [11]:
X = df['URL']
Y = df['label']

In [12]:
vectorizer = TfidfVectorizer(max_features=600)
vectorizer.fit(X)
X = vectorizer.transform(X)

In [13]:
x_train, x_test, y_train, y_test = train_test_split(X, Y, test_size=0.25, stratify=Y, random_state=1)

In [14]:
print("Shape of original dataset:", X.shape)
print("Shape of train and test sets:", x_train.shape, x_test.shape)

Shape of original dataset: (235795, 600)
Shape of train and test sets: (176846, 600) (58949, 600)


In [15]:
smote = SMOTE(random_state=42)
x_train_resampled, y_train_resampled = smote.fit_resample(x_train, y_train)

In [16]:
print("Shape of training set after SMOTE:", x_train_resampled.shape, y_train_resampled.shape)
print("Class distribution after SMOTE:")
print(pd.Series(y_train_resampled).value_counts())

Shape of training set after SMOTE: (202274, 600) (202274,)
Class distribution after SMOTE:
label
0    101137
1    101137
Name: count, dtype: int64


In [17]:
from sklearn.linear_model import LogisticRegression

In [18]:
model1 = LogisticRegression(max_iter=200, random_state=0)

In [19]:
model1.fit(x_train_resampled, y_train_resampled)

LogisticRegression(max_iter=200, random_state=0)

In [20]:
x_train_predict = model1.predict(x_train)
training_data_accuracy = accuracy_score(x_train_predict, y_train)
print("Training Data Accuracy:", training_data_accuracy)

Training Data Accuracy: 0.9942605430713728


In [21]:
x_test_predict = model1.predict(x_test)
test_data_accuracy = accuracy_score(x_test_predict, y_test)
print("Test Data Accuracy:", test_data_accuracy)

Test Data Accuracy: 0.9947412169841727


In [22]:
training_precision = precision_score(y_train, x_train_predict)
print('Training data precision score = ', training_precision)
training_recall = recall_score(y_train, x_train_predict)
print('Training data recall score = ', training_recall)
training_matrix = confusion_matrix(y_train, x_train_predict)
print('Training data confusion matrix = ', training_matrix)

Training data precision score =  0.9920386860883411
Training data recall score =  0.9979730464617301
Training data confusion matrix =  [[ 74899    810]
 [   205 100932]]


In [23]:
testing_precision = precision_score(y_test, x_test_predict)
print('Testing data precision score = ', testing_precision)
testing_recall = recall_score(y_test, x_test_predict)
print('Testing data recall score = ', testing_recall)
testing_matrix = confusion_matrix(y_test, x_test_predict)
print('Testing data confusion matrix = ', testing_matrix)

Testing data precision score =  0.9926550839208282
Testing data recall score =  0.9981906089639012
Testing data confusion matrix =  [[24987   249]
 [   61 33652]]


In [24]:
def predict_text(input_text):
    # Transform the input text using the fitted vectorizer
    input_vector = vectorizer.transform([input_text])

    # Make the prediction
    prediction = model1.predict(input_vector)

    return prediction[0]

In [25]:
input_text = "www.google.com"
predicted_class = predict_text(input_text)
print(f"The predicted class for the input text is: {predicted_class}")


The predicted class for the input text is: 1


In [26]:
input_text = "https://www.uni-mainz.de"
predicted_class = predict_text(input_text)
print(f"The predicted class for the input text is: {predicted_class}")

The predicted class for the input text is: 1


In [27]:
input_text = "http://google.com-redirect@valimail.com"
predicted_class = predict_text(input_text)
print(f"The predicted class for the input text is: {predicted_class}")

The predicted class for the input text is: 0
